In [1]:
sampling_method_type = ["over", "under"]
sampling_method_name = {
    "over": ["random", "smote", "adasyn", "borderline_smote", "kmeans_smote", "svmsmote"],
    "under": ["random", "cluster_centroid", "edited_nearest_neighbors", "repeated_edited_nearest_neighbors", "all_knn", "instance_hardness_threshold", "near_miss", "neighbourhood_cleaning_rule", "one_sided_selection", "tomek_links"]
}
sampling_method_limit_place = [1, 2, 3, 4, 5]


In [2]:
data_path_train = ["data/train/raw", "data/train/binary", "data/train/sampled_raw", "data/train/sampled_binary"]
data_path_test = ["data/test/raw", "data/test/binary"]


| sampling.method.type |      sampling.method.name       | sampling.method.limit_place |                                     data_path.train, data_path.test                     |
| -------------------- | ------------------------------- | --------------------------- | --------------------------------------------------------------------------------------- |
|        "over"        |   sampling_method_name["over"]  |      [1, 2, 3, 4, 5]        |  ("data/train/sampled_raw", "data/test/raw"), ("data/train/binary", "data/test/binary") |
|       "under"        |  sampling_method_name["under"]  |      [1, 2, 3, 4, 5]        |  ("data/train/sampled_raw", "data/test/raw"), ("data/train/binary", "data/test/binary") |

```bash
dvc exp run --queue -S 'sampling.method.type="over"' -S ...

In [3]:
#!/usr/bin/env python3
# save as: tools/queue_sampling_experiments.py

import subprocess
from itertools import product

# 表に基づく定義
sampling_method_type = ["over", "under"]
sampling_method_name = {
    "over": [
        "random", "smote", "adasyn",
        "borderline_smote", "kmeans_smote", "svmsmote",
    ],
    "under": [
        "random", "cluster_centroid", "edited_nearest_neighbors",
        "repeated_edited_nearest_neighbors", "all_knn",
        "instance_hardness_threshold", "near_miss",
        "neighbourhood_cleaning_rule", "one_sided_selection",
        "tomek_links",
    ],
}
sampling_method_limit_place = [1, 2, 3, 4, 5]

# 表の2行目にある data_path.train/test の組
data_path_pairs = [
    ("data/train/sampled_raw", "data/test/raw"),
    ("data/train/binary", "data/test/binary"),
]

def build_cmd(stype: str, sname: str, limit_place: int, train_path: str, test_path: str):
    return [
        "dvc", "exp", "run", "--queue",
        "-S", f'sampling.method.type="{stype}"',
        "-S", f'sampling.method.name="{sname}"',
        "-S", f"sampling.method.limit_place={limit_place}",
        "-S", f"data_path.train={train_path}",
        "-S", f"data_path.test={test_path}",
    ]

def main(dry_run: bool = True):
    cmds = []
    for stype in sampling_method_type:
        for sname, limit_place, (train_path, test_path) in product(
            sampling_method_name[stype],
            sampling_method_limit_place,
            data_path_pairs,
        ):
            cmds.append(build_cmd(stype, sname, limit_place, train_path, test_path))

    # 出力
    print(f"# total queued experiments: {len(cmds)}")
    for c in cmds:
        print(" ".join(c))

    # if not dry_run:
    #     # 実キュー投入
    #     for c in cmds:
    #         subprocess.run(c, check=True)

        # すべて実行（必要なら有効化）
        # subprocess.run(["dvc", "exp", "run", "--run-all"], check=True)

if __name__ == "__main__":
    # dry_run=True: コマンド出力のみ（安全）
    # dry_run=False: 実際にキュー投入
    main(dry_run=True)

# total queued experiments: 160
dvc exp run --queue -S sampling.method.type="over" -S sampling.method.name="random" -S sampling.method.limit_place=1 -S data_path.train=data/train/sampled_raw -S data_path.test=data/test/raw
dvc exp run --queue -S sampling.method.type="over" -S sampling.method.name="random" -S sampling.method.limit_place=1 -S data_path.train=data/train/binary -S data_path.test=data/test/binary
dvc exp run --queue -S sampling.method.type="over" -S sampling.method.name="random" -S sampling.method.limit_place=2 -S data_path.train=data/train/sampled_raw -S data_path.test=data/test/raw
dvc exp run --queue -S sampling.method.type="over" -S sampling.method.name="random" -S sampling.method.limit_place=2 -S data_path.train=data/train/binary -S data_path.test=data/test/binary
dvc exp run --queue -S sampling.method.type="over" -S sampling.method.name="random" -S sampling.method.limit_place=3 -S data_path.train=data/train/sampled_raw -S data_path.test=data/test/raw
dvc exp run --que